# Agent Platform — Hands On

**Problem Type A:** *"Design an AI agent platform for non-technical users to configure workflow
automations across multiple channels."*

This notebook builds and runs every piece of it: channel adapters, the tool registry, staged
rollout, routing and conflict resolution, the guardrail decision engine, and the orchestrator's two
core properties - idempotency and durability. No LLM calls anywhere - every decision is
deterministic, so every cell below is fast and exactly reproducible.

Read `docs/01-theory.md` first if you have not.

In [8]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "agent_platform").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

print("project root:", ROOT)

project root: d:\INTERVIEW PREPARATION\DevRev_Preparation\System_Design_Round\agent_platform


---
# Part 1 — Channel adapters: one canonical Event

Wildly different raw payload shapes, normalised once, at the edge, into the one shape everything
downstream ever sees. Nothing past this point branches on which channel an event came from.

In [9]:
from agent_platform import channels

webhook_event = channels.from_webhook({
    "type": "ticket.created", "id": "wh_1", "tenant_id": "cascade_robotics",
    "ticket_id": "TCK-5510", "order_id": "ORD-2201", "refund_amount_usd": 35.0,
})
slack_event = channels.from_slack({
    "team_id": "cascade_robotics", "ts": "1699999999.001", "text": "prod checkout is failing!",
    "is_urgent": True, "channel_name": "support-escalations",
})

for label, ev in [("webhook", webhook_event), ("slack", slack_event)]:
    print(f"{label:<10}channel={ev.channel.value:<10}event_type={ev.event_type:<18}"
          f"target={ev.target_entity_id}")

webhook   channel=webhook   event_type=ticket.created    target=TCK-5510
slack     channel=slack     event_type=urgent_message    target=1699999999.001


---
# Part 2 — The tool registry: constrained schemas, not free text

"Prefer constrained tool schemas and enums over free-text arguments; validate every tool argument
before execution" (§3.4). A malformed call is rejected before anything executes.

In [10]:
from agent_platform.tools import REGISTRY, validate_args

for name, tool in REGISTRY.items():
    print(f"{name:<16}destructive={tool.destructive!s:<6}schema={tool.schema}")

draft_reply     destructive=False schema={'ticket_id': 'str', 'body': 'str'}
issue_refund    destructive=True  schema={'order_id': 'str', 'amount_usd': 'float'}
close_ticket    destructive=True  schema={'ticket_id': 'str'}
tag_ticket      destructive=False schema={'ticket_id': 'str', 'tag': 'str'}


In [11]:
tool = REGISTRY["issue_refund"]

# Wrong type - rejected before execution.
d = validate_args(tool, {"order_id": "ORD-1", "amount_usd": "fifty dollars"})
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# Missing a required field - rejected before execution.
d = validate_args(tool, {"order_id": "ORD-1"})
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# Valid.
d = validate_args(tool, {"order_id": "ORD-1", "amount_usd": 35.0})
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

DENY [type_mismatch] 'amount_usd' expected float, got str
DENY [missing_required_args] missing: ['amount_usd']
ALLOW [args_valid] arguments match the tool's schema


---
# Part 3 — Staged rollout: draft -> testing -> shadow -> live -> autonomous

Promotion is role-gated and order-enforced - the same shape as `delivery_framework_platform`'s
stage gates. The author who wrote a workflow can never promote it themselves.

**Function-by-function explanation of the cell below:**

- `get_principal(user_id)` — looks up a hardcoded demo `Principal` (id, display name, role) by user id; raises `KeyError` if unknown.
- `build_store()` — creates a `WorkflowStore` and publishes three demo `WorkflowSpec`s into it (`wf_ticket_triage`, `wf_legacy_ticket_tagger`, `wf_runaway`), returning the populated store.
- `default_policy()` — returns a demo `GuardrailPolicy` for the tenant with a $50 spend cap, a max of 10 steps, and `tag_ticket` allow-listed as a destructive tool for autonomous runs.
- `promote(store, workflow_id, to_status, signer)` — advances a workflow's latest version to the next rollout stage; denies if the workflow is unknown, if `signer` isn't `admin`/`approver` (so an author can't self-promote), or if `to_status` would skip a stage — otherwise mutates the spec's status and returns an "allowed" `Decision`.

Everything else in the cell (`store`, `policy`, `author`, `wrong_hat`, `approver`, `admin`) are just the return values of the calls above, and `WorkflowStatus` is the enum ordering the rollout stages (`DRAFT → TESTING → SHADOW → LIVE → AUTONOMOUS`).

In [15]:
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from _scenario import build_store, default_policy, TENANT

from agent_platform.identity import get_principal
from agent_platform.models import WorkflowStatus
from agent_platform.workflows import promote

store = build_store()
policy = default_policy()
author = get_principal("u_author_dana")
wrong_hat = get_principal("u_author_wrong_hat")
approver = get_principal("u_approver_raj")
admin = get_principal("u_admin_lee")

# The author cannot promote their own workflow.
d = promote(store, "wf_ticket_triage", WorkflowStatus.TESTING, wrong_hat)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# Cannot skip a stage.
d = promote(store, "wf_ticket_triage", WorkflowStatus.AUTONOMOUS, admin)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# The correct path, one stage at a time.
for target in [WorkflowStatus.TESTING, WorkflowStatus.SHADOW, WorkflowStatus.LIVE]:
    d = promote(store, "wf_ticket_triage", target, admin)
    print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

DENY [wrong_role] 'author' may not promote a workflow; requires admin or approver
DENY [cannot_skip_stage] 'wf_ticket_triage' is at DRAFT; can only promote to TESTING, not AUTONOMOUS
OK [promoted] 'wf_ticket_triage' is now TESTING
OK [promoted] 'wf_ticket_triage' is now SHADOW
OK [promoted] 'wf_ticket_triage' is now LIVE


---
# Part 4 — Routing: priority and the entity lock

Two independent mechanisms. Priority is a *design-time* choice - which workflow should win a
same-trigger conflict. The entity lock is a *run-time* safety property - can two workflows ever be
mutating the same ticket at once - and it holds even if priority was misconfigured.

**Function-by-function explanation of the cell below:**

- `channels.from_webhook(payload)` — normalises the raw webhook payload into a canonical `Event` (same adapter used in Part 1); here it builds a second `ticket.created` event for `TCK-5510`.
- `store.all_live(TENANT)` — returns the latest-version `WorkflowSpec` for every workflow this tenant has that is currently `LIVE` (both `wf_ticket_triage`, promoted to `LIVE` in Part 3, and `wf_legacy_ticket_tagger`, published `LIVE` from the start).
- `routing.route(event, live_workflows)` — picks exactly one workflow for the event, or explains why none ran:
  - internally calls `matching_workflows()`, which filters to workflows for the same tenant, not in `DRAFT`, with a trigger matching the event's `channel` + `event_type`;
  - denies `no_trigger_match` if nothing matches, or `entity_locked` if the event's target entity already has an active run;
  - otherwise picks the **highest-priority** matching trigger as the winner and returns an "allowed" `Decision` naming it.

Since both `wf_ticket_triage` (priority `10`) and `wf_legacy_ticket_tagger` (priority `1`) match the same `webhook` / `ticket.created` trigger, `route()` deterministically picks `wf_ticket_triage` — priority breaks the tie, not registration order or any randomness.

In [20]:
status_wf_legacy_ticket_tagger = store.latest("wf_legacy_ticket_tagger")
status_wf_ticket_triage = store.latest("wf_ticket_triage")
print(status_wf_legacy_ticket_tagger.status)
print(status_wf_ticket_triage.status)

WorkflowStatus.LIVE
WorkflowStatus.LIVE


In [21]:
from agent_platform import routing

event = channels.from_webhook({
    "type": "ticket.created", "id": "wh_2", "tenant_id": TENANT,
    "ticket_id": "TCK-5510", "order_id": "ORD-2201", "refund_amount_usd": 35.0,
})
live_workflows = store.all_live(TENANT)
d = routing.route(event, live_workflows)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")
print("(the high-priority ticket-triage workflow wins over the low-priority legacy tagger,")
print(" even though both match the same trigger)")

OK [routed] routed to 'wf_ticket_triage' v1
(the high-priority ticket-triage workflow wins over the low-priority legacy tagger,
 even though both match the same trigger)


In [1]:
# The exclusivity lock is independent of priority.
assert routing.acquire_lock("TCK-locked-demo", "run_a") is True
locked_again = routing.acquire_lock("TCK-locked-demo", "run_b")
print(f"second concurrent acquire on the same ticket: {locked_again}")
routing.release_lock("TCK-locked-demo")
print(f"after release, acquire succeeds again: {routing.acquire_lock('TCK-locked-demo', 'run_c')}")
routing.release_lock("TCK-locked-demo")

second concurrent acquire on the same ticket: False
after release, acquire succeeds again: True


---
# Part 5 — The guardrail decision

Five ordered rules, deny overrides - the same shape as `authz.policy.decide()` in the RAG project
and `gates.py::sign_off()` in the delivery framework.

In [1]:
from agent_platform import orchestrator
from agent_platform.guardrails import authorize_step
from agent_platform.models import Run

refund_tool = REGISTRY["issue_refund"]

def probe(workflow, cost=0.0, approval=None, **run_kwargs):
    run = Run(run_id="probe", workflow_id=workflow.workflow_id, workflow_version=1, event=event)
    step = workflow.steps[1]   # the refund step
    return authorize_step(workflow, run, step, refund_tool, cost, policy, approval=approval)

live_wf = store.get_version("wf_ticket_triage", 1)
d = probe(live_wf)
print(f"LIVE, no approver          -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")
d = probe(live_wf, approval=approver)
print(f"LIVE, with approver        -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")
d = probe(live_wf, cost=999.0, approval=approver)
print(f"LIVE, over the spend cap   -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")

LIVE, no approver          -> DENY [needs_human_approval]
LIVE, with approver        -> ALLOW [human_approved]
LIVE, over the spend cap   -> DENY [spend_cap_exceeded]


---
# Part 6 — The full run: a real destructive step, paused, then approved

In [1]:
run = orchestrator.run_workflow("nb_run_001", live_wf, event, policy)
print(f"state after first pass: {run.state.value}")

run = orchestrator.resume(run, live_wf, policy, approval=approver)
print(f"state after approval:   {run.state.value}  total_cost=${run.total_cost_usd:.2f}")

state after first pass: paused_for_approval
state after approval:   completed  total_cost=$35.02


---
# Part 7 — Idempotency: a retry never double-applies a side effect

In [1]:
before = orchestrator.external_call_count("issue_refund")
run.next_step_index = 1   # rewind the checkpoint to re-attempt the refund step only
run = orchestrator.resume(run, live_wf, policy, approval=approver)
after = orchestrator.external_call_count("issue_refund")
print(f"issue_refund real calls before retry: {before}")
print(f"issue_refund real calls after retry:  {after}")
print(f"the retried step's side_effect_applied flag: {run.completed_steps[-1].side_effect_applied}")

issue_refund real calls before retry: 1
issue_refund real calls after retry:  1
the retried step's side_effect_applied flag: False


---
# Part 8 — Durability: a simulated crash, then resume from the checkpoint

In [1]:
event2 = channels.from_webhook({
    "type": "ticket.created", "id": "wh_3", "tenant_id": TENANT,
    "ticket_id": "TCK-5599", "order_id": "ORD-2299", "refund_amount_usd": 8.0,
})
run2 = orchestrator.run_workflow("nb_run_002", live_wf, event2, policy, crash_after_step=0)
print(f"after simulated crash: state={run2.state.value}  next_step_index={run2.next_step_index}")

run2 = orchestrator.resume(run2, live_wf, policy, approval=approver)
draft_executions = [e for e in run2.events if e["kind"] == "step_executed" and e["step"] == "draft"]
print(f"state after resume: {run2.state.value}")
print(f"'draft' step executed {len(draft_executions)} time(s) total - never re-ran after the crash")

after simulated crash: state=crashed  next_step_index=1
state after resume: completed
'draft' step executed 1 time(s) total - never re-ran after the crash


---
# Part 9 — Observability: the full, replayable trace

In [1]:
from agent_platform.observability import render_run

print(render_run(run2))

run nb_run_002  workflow=wf_ticket_triage v1  state=completed  cost=$8.02
------------------------------------------------------------------------------------------------
  step 0  step_authorize  draft           OK                        no confirmation gate on a non-destructive step
  step 0  step_executed   draft           APPLIED                   $0.02
  step 1  *** CRASH after 'draft' ***
  step 1  step_authorize  refund          OK                        approved by Raj (Support Lead, approver)
  step 1  step_executed   refund          APPLIED                   $8.00
  step 2  RUN COMPLETE


---
# Part 10 — What to take away

1. **The model decides THAT something should happen; a typed, validated schema decides HOW.**
   Constrained arguments, checked before execution, are where determinism actually lives.
2. **Autonomous status raises the ceiling; it never removes it.** A non-allow-listed destructive
   tool still needs a human even on a fully autonomous workflow, and the spend cap applies at every
   stage.
3. **One execution loop, not two.** `resume()` and a fresh run share the exact same code path -
   there's no separate "recovery" path with its own bugs.
4. **An idempotency key belongs to the action, not the run.** A run can legitimately retry; one
   specific side effect must never apply twice.
5. **Priority and the entity lock answer two different questions.** Which workflow should win a
   conflict, versus whether two can ever run concurrently - independent checks, on purpose.
6. **A halted run with a named reason beats a workflow that loops until someone notices.**

Next: `../INTERVIEW_SCRIPT.md` - how to present all of this on a whiteboard in 60 minutes.